In [1]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

load_dotenv()
CSV_PATH = Path(os.getenv("PHYSIONET_DIR")) / "hemorrhage_diagnosis_raw_ct.csv"

In [2]:
df: pd.DataFrame = pd.read_csv(CSV_PATH)
df.head()

,PatientNumber,SliceNumber,Intraventricular,Intraparenchymal,Subarachnoid,Epidural,Subdural,No_Hemorrhage,Fracture_Yes_No
0,49,1,0,0,0,0,0,1,0
1,49,2,0,0,0,0,0,1,0
2,49,3,0,0,0,0,0,1,0
3,49,4,0,0,0,0,0,1,0
4,49,5,0,0,0,0,0,1,0


In [3]:
num_patients: int = df["PatientNumber"].nunique()
print(f"Number of unique patients: {num_patients}")

Number of unique patients: 75


In [4]:
hemorrhage_patients = df[df["No_Hemorrhage"] == 0]["PatientNumber"].nunique()

print(f"Number of patients with hemorrhage: {hemorrhage_patients}")
print(f"Number of patients without hemorrhage: {num_patients - hemorrhage_patients}")

Number of patients with hemorrhage: 36
Number of patients without hemorrhage: 39


In [5]:
hemorrhage_types: list[str] = [
    "Intraparenchymal",
    "Intraventricular",
    "Subarachnoid",
    "Subdural",
    "Epidural",
]
print(df[hemorrhage_types].sum(axis=1).value_counts())

0    2496
1     293
2      24
3       1
Name: count, dtype: int64


In [6]:
for hemorrhage_type in hemorrhage_types:
    mask: pd.Series = df[hemorrhage_type] == 1
    count: int = df[mask].shape[0]
    print(f"{hemorrhage_type:<20}: {count:<3} * 20% = {count * 0.2:.1f}")

Intraparenchymal    : 73  * 20% = 14.6
Intraventricular    : 24  * 20% = 4.8
Subarachnoid        : 18  * 20% = 3.6
Subdural            : 56  * 20% = 11.2
Epidural            : 173 * 20% = 34.6


In [7]:
# Print the slice with multiple hemorrhage types
for patient in df["PatientNumber"].unique():
    patient_df: pd.DataFrame = df[df["PatientNumber"] == patient]
    if patient_df[hemorrhage_types].sum(axis=1).max() > 1:
        print(
            f"Patient {patient} has multiple hemorrhage types:",
            *patient_df[patient_df[hemorrhage_types].sum(axis=1) > 1][
                "SliceNumber"
            ].values,
        )

Patient 49 has multiple hemorrhage types: 21
Patient 71 has multiple hemorrhage types: 12 13 14 15
Patient 76 has multiple hemorrhage types: 38 39
Patient 80 has multiple hemorrhage types: 8 9 10 14 15
Patient 91 has multiple hemorrhage types: 18 19
Patient 92 has multiple hemorrhage types: 15 19 20 22
Patient 93 has multiple hemorrhage types: 18
Patient 94 has multiple hemorrhage types: 22 23 24 25
Patient 97 has multiple hemorrhage types: 20 21


In [8]:
hemorrhage_patient_df: pd.DataFrame = pd.concat(
    [
        pd.DataFrame(
            {
                "HemorrhageType": hemorrhage_type,
                "PatientNumber": df.loc[
                    df[hemorrhage_type] == 1, "PatientNumber"
                ].unique(),
            }
        )
        for hemorrhage_type in hemorrhage_types
    ],
    ignore_index=True,
)

for hemorrhage_type in hemorrhage_types:
    patients = hemorrhage_patient_df[
        hemorrhage_patient_df["HemorrhageType"] == hemorrhage_type
    ]["PatientNumber"].unique()
    print("List of patients with", hemorrhage_type, "hemorrhage:", *patients)

List of patients with Intraparenchymal hemorrhage: 49 50 51 53 58 69 71 72 76 79 80 84 91 92 94 97
List of patients with Intraventricular hemorrhage: 80 85 91 92 94
List of patients with Subarachnoid hemorrhage: 76 80 82 84 92 93 94
List of patients with Subdural hemorrhage: 51 71 74 81
List of patients with Epidural hemorrhage: 49 52 53 66 67 68 70 73 74 75 77 78 83 86 87 88 89 90 92 93 97
